
# 12 — Data Imputation and Encoding

**Scope:** The missing and incorrect values are handled, categorical features are encoded.


Since we decided to predict missing and incorrect values, we have to split the training data into training and validation sets before applying imputation and encoding.

# Table of Contents

Take this as an example for a Table of Contents for your notebook.
We have to fix all the names and sections according to what we actually do in the notebook.

<a class="anchor" id="top"></a>

** **

1. [Importing Libraries & Data](#1.-Importing-Libraries-&-Data) <br><br>
    
2. [Exploratory Data Analysis](#2.-Exploratory-Data-Analysis)
    
   2.1 [Incoherencies](#2.1-Incoherencies) <br>
   
   &emsp; 2.1.1 [Address Identified Incoherencies](#2.1.1-Address-Identified-Incoherencies) <br><br>
    
3. [Data Cleaning & Preprocessing](#3.-Data-Cleaning-&-Preprocessing)

   3.1 [Duplicates](#3.1-Duplicates) <br>
    
   3.2 [Feature Engineering](#3.2-Feature-Engineering) <br>
   
   &emsp; 3.2.1 [Data Type Conversions](#3.2.1-Data-Type-Conversions) <br>
   
   &emsp; 3.2.2 [Encoding](#3.2.2-Encoding) <br>
   
   &emsp; 3.2.3 [Other Transformations](#3.2.3-Other-Transformations) <br>
    
   &emsp; 3.2.4 [Unique Feature-Pair Analysis](#3.2.4-Unique-Feature-Pair-Analysis) <br> 

   3.3 [Train-Test Split](#3.3-Train-Test-Split) <br>
   
   3.4 [Missing Values](#3.4-Missing-Values) <br>
    
   3.5 [Outliers](#3.5-Outliers) <br>

   3.6 [Visualisations](#3.6-Visualisations) <br><br>
   

In [263]:
import os, re, math, warnings
from pathlib import Path
from datetime import datetime
import json
import pandas as pd
import numpy as np
import re
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("mode.copy_on_write", True)
warnings.filterwarnings("ignore")

RANDOM_STATE = 42  # for reproducibility of any sampling


In [264]:
# Load the data paths
data_dir = "../data/"

# Load the raw data into a pandas dataframe
df = pd.read_csv(os.path.join(data_dir, "processed_data/11_processed_train_data.csv"))
X_test = pd.read_csv(os.path.join(data_dir, "test.csv"))

# Drop the column carID from both dataframes because it is not needed for modeling
if "carID" in df.columns:
    df = df.drop(columns=["carID"])
if "carID" in X_test.columns:
    x_test = X_test.drop(columns=["carID"])


print("Loaded shape:", df.shape)
display(df.head(3))

print("Loaded test shape:", X_test.shape)
display(X_test.head(3))


Loaded shape: (75973, 13)


,Brand,model,year,price,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,Volkswagen,Golf,2016.0,22290.0,Semi-Auto,28421.0,Petrol,NaN,11.417268,2.0,63.0,4.0,0.0
1,Toyota,Yaris,2019.0,13790.0,Manual,4589.0,Petrol,145.0,47.900000,1.5,50.0,1.0,0.0
2,Audi,Q2,2019.0,24990.0,Semi-Auto,3624.0,Petrol,145.0,40.900000,1.5,56.0,4.0,0.0


Loaded test shape: (32567, 13)


,carID,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
0,89856,Hyundai,I30,2022.878006,Automatic,30700.000000,petrol,205.0,41.5,1.6,61.0,3.0,0.0
1,106581,VW,Tiguan,2017.000000,Semi-Auto,-48190.655673,Petrol,150.0,38.2,2.0,60.0,2.0,0.0
2,80886,BMW,2 Series,2016.000000,Automatic,36792.000000,Petrol,125.0,51.4,1.5,94.0,2.0,0.0


### Split the Training data into training and validation sets

In [265]:
# ======================================================
# Split df into Training and Validation Set (60/20/20 total)
# ======================================================

from sklearn.model_selection import train_test_split

# Separate features and target from the training data
X = df.drop(columns=["price"])
y = df["price"]

# Split df (80% of total data) into training and validation
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.25,        # 25% of 80% train = 20% of total
    random_state=42,
)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_val shape:   {X_val.shape}")
print(f"y_val shape:   {y_val.shape}")
print(f"X_test shape:  {X_test.shape}")


X_train shape: (56979, 12)
y_train shape: (56979,)
X_val shape:   (18994, 12)
y_val shape:   (18994,)
X_test shape:  (32567, 13)


# Handling Missing Values

- Numerical: fill with the median
- Categorical: fill with the most frequent value

In [266]:
# ======================================================
# Missing-Value-Report (no leakage)
# ======================================================
import pandas as pd

def missing_report(X: pd.DataFrame, name: str) -> pd.DataFrame:
    mv = X.isna().sum()
    mv = mv[mv > 0].sort_values(ascending=False)
    if mv.empty:
        print(f"[{name}] No missing values found. (n_rows={len(X)})")
        return pd.DataFrame(columns=["n_missing", "pct_missing"])
    pct = (mv / len(X) * 100).round(2)
    report = pd.DataFrame({"n_missing": mv, "pct_missing": pct})
    print(f"[{name}] Missing Values (n_rows={len(X)}):")
    display(report)
    return report

# Reports for the splits
mv_train = missing_report(X_train, "X_train")
mv_val   = missing_report(X_val,   "X_val")
mv_test  = missing_report(X_test,  "X_test")

# Optional: warning if columns are missing only in val/test (but not in train)
cols_mv_train = set(mv_train.index) if not mv_train.empty else set()
cols_mv_val   = set(mv_val.index)   if not mv_val.empty   else set()
cols_mv_test  = set(mv_test.index)  if not mv_test.empty  else set()

only_val = cols_mv_val - cols_mv_train
only_test = cols_mv_test - cols_mv_train
if only_val:
    print("⚠️ Columns with missing values only in X_val:", sorted(only_val))
if only_test:
    print("⚠️ Columns with missing values only in X_test:", sorted(only_test))

# Helpful for the next step (imputation/encoding):
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print(f"#numeric_cols: {len(numeric_cols)} → {numeric_cols[:10]}{' ...' if len(numeric_cols) > 10 else ''}")
print(f"#categorical_cols: {len(categorical_cols)} → {categorical_cols[:10]}{' ...' if len(categorical_cols) > 10 else ''}")


[X_train] Missing Values (n_rows=56979):


,n_missing,pct_missing
mpg,7123,12.50
tax,6251,10.97
engineSize,1583,2.78
mileage,1372,2.41
transmission,1161,2.04
previousOwners,1144,2.01
Brand,1144,2.01
paintQuality%,1141,2.00
hasDamage,1141,2.00
fuelType,1136,1.99


[X_val] Missing Values (n_rows=18994):


,n_missing,pct_missing
mpg,2291,12.06
tax,2031,10.69
engineSize,508,2.67
mileage,460,2.42
fuelType,430,2.26
hasDamage,407,2.14
previousOwners,406,2.14
model,390,2.05
paintQuality%,383,2.02
Brand,377,1.98


[X_test] Missing Values (n_rows=32567):


,n_missing,pct_missing
tax,3308,10.16
mpg,3288,10.10
mileage,689,2.12
fuelType,656,2.01
year,653,2.01
model,650,2.00
Brand,649,1.99
engineSize,628,1.93
paintQuality%,625,1.92
transmission,623,1.91


#numeric_cols: 8 → ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'paintQuality%', 'previousOwners', 'hasDamage']
#categorical_cols: 4 → ['Brand', 'model', 'transmission', 'fuelType']


In [267]:
def impute_transmission(df, min_model_count=20):
    # Make a copy so the original dataframe isn't modified directly
    df = df.copy()
    # Count how many entries exist per model
    model_counts = df['model'].value_counts()

    # Keep only models with enough data points
    valid_models = model_counts[model_counts >= min_model_count].index

    # Loop through those models and fill NaN with the most common value (mode)
    for model in valid_models:
        mask = (df['model'] == model) & (df['transmission'].isna())
        mode_values = df.loc[df['model'] == model, 'transmission'].mode()
        if not mode_values.empty:
            df.loc[mask, 'transmission'] = mode_values[0]

    # Check how many missing values are still left
    remaining_nas = df['transmission'].isna().sum()
    
    if remaining_nas > 0:
        # Encode categorical variables numerically
        # RandomForest can only work with numeric data
        for col in ['Brand', 'model', 'fuelType']:
            means = df.loc[df['transmission'].notna()].groupby(col)['price'].mean()
            df[col] = df[col].map(means) # for test data purposes, we only use infos from the traindata

        # Split into training (known transmission) and test (missing transmission)
        transmission_train = df[df['transmission'].notna()]
        test = df[df['transmission'].isna()]

        # Select predictor features
        features = ['Brand', 'model', 'fuelType', 'engineSize', 'year', 'mpg', 'tax']
        X_train = transmission_train[features]
        y_train = transmission_train['transmission']
        X_test = test[features]

        # Train a Random Forest classifier to predict transmission type
        model = RandomForestClassifier(
            n_estimators=200,  # number of trees
            max_depth=10,      # limit depth to avoid overfitting
            random_state=42
        )
        model.fit(X_train, y_train)

        # Predict missing transmission values
        preds = model.predict(X_test)
        df.loc[df['transmission'].isna(), 'transmission'] = preds
        return df

In [268]:
def impute_fuelType(df, min_model_count=20, is_test_data=True):
    
    # work on a copy
    df = df.copy()

    # per-model mode fill (only for models with enough rows)
    counts = df['model'].value_counts()
    valid_models = counts[counts >= min_model_count].index

    for m in valid_models:
        mask_missing = (df['model'] == m) & (df['fuelType'].isna())
        mode_vals = df.loc[df['model'] == m, 'fuelType'].mode()
        if not mode_vals.empty:
            df.loc[mask_missing, 'fuelType'] = mode_vals[0]

    # if still missing, train a classifier
    remaining = df['fuelType'].isna().sum()
    if remaining > 0:
        # encode categorical predictors
        if is_test_data:
            predictors = ['Brand', 'model', 'transmission', 'engineSize', 'year', 'mpg', 'price']
        else:
            predictors = ['Brand', 'model', 'transmission', 'engineSize', 'year', 'mpg']
        for col in predictors:
            if df[col].dtype == 'object':
                means = df.loc[df['fuelType'].notna()].groupby(col)['price'].mean()
                df[col] = df[col].map(means) # for test data purposes, we only use infos from the traindata

        # split into known vs missing target
        fuelType_train = df[df['fuelType'].notna()]
        test  = df[df['fuelType'].isna()]

        X_train = fuelType_train[predictors]
        y_train = fuelType_train['fuelType']
        X_test  = test[predictors]

        # encode target
        y_le = LabelEncoder()
        y_train_enc = y_le.fit_transform(y_train.astype(str))

        # train classifier
        clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=77)
        clf.fit(X_train, y_train_enc)

        # predict and inverse-transform
        preds_enc = clf.predict(X_test)
        preds = y_le.inverse_transform(preds_enc)

        # fill back
        df.loc[df['fuelType'].isna(), 'fuelType'] = preds

    return df

In [269]:
def impute_numeric_rf(df, target_col, rf_features):
    """
    Fill missing numeric values in a column using a RandomForestRegressor.
    """

    # Copy the DataFrame so the original one is not changed
    df = df.copy()

    # If there are no missing values, just return the DataFrame
    if not df[target_col].isna().any():
        return df

    # Encode categorical features so the model can handle them
    for col in rf_features:
        if df[col].dtype == 'object':
            means = df.loc[df[target_col].notna()].groupby(col)['price'].mean()
            df[col] = df[col].map(means) # for test data purposes, we only use infos from the traindata

    # Split data into known and missing target values
    train = df[df[target_col].notna()]
    test = df[df[target_col].isna()]

    X_train = train[rf_features]
    y_train = train[target_col]
    X_test = test[rf_features]

    # Train the Random Forest model
    # n_estimators=300 number of trees in the forest, more trees → better accuracy, but slower
    # max_depth=12 maximum depth of each tree, limits how detailed (complex) each tree can get → prevents overfitting
    # random_state=23 seed for randomness, makes results reproducible (same every time you run it)
    model = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=23)
    model.fit(X_train, y_train)

    # Predict missing values and fill them in
    preds = model.predict(X_test)
    df.loc[df[target_col].isna(), target_col] = preds

    return df

In [270]:
def impute_brand (df, min_model_count=20, inlucePrice=True):
    
    # work on a copy
    df = df.copy()

    # per-model mode fill (only for models with enough rows)
    counts = df['model'].value_counts()
    valid_models = counts[counts >= min_model_count].index

    # Mapping: Modell → häufigste Marke
    df2=df.copy()
    model_to_brand = (
        df2.dropna(subset=['Brand', 'model'])
          .groupby('model')['Brand']
          .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
    )

    # Fehlende Marken füllen, wo Modell bekannt ist
    # sichere Kopie der Masken-Logik, NUR mit df2
    mask = df['Brand'].isna() & df['model'].notna()

    # benutze df2 für das Mapping (niemals df direkt!)
    df.loc[mask, 'Brand'] = df2.loc[mask, 'model'].map(model_to_brand)

    # if still missing, train a classifier
    remaining = df['Brand'].isna().sum()
    if remaining > 0:
        # encode categorical predictors
        if inlucePrice:
            predictors = ['transmission', 'engineSize', 'fuelType', 'mpg', 'price']
        else:
            predictors = ['transmission', 'engineSize', 'fuelType', 'mpg']
        for col in predictors:
            if df[col].dtype == 'object':
                means = df.loc[df["Brand"].notna()].groupby(col)['price'].mean()
                df[col] = df[col].map(means) # for test data purposes, we only use infos from the traindata

        # split into known vs missing target
        fuelType_train = df[df['Brand'].notna()]
        test  = df[df['Brand'].isna()]

        X_train = fuelType_train[predictors]
        y_train = fuelType_train['Brand']
        X_test  = test[predictors]

        # encode target
        y_le = LabelEncoder()
        y_train_enc = y_le.fit_transform(y_train.astype(str))

        # train classifier
        clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=77)
        clf.fit(X_train, y_train_enc)

        # predict and inverse-transform
        preds_enc = clf.predict(X_test)
        preds = y_le.inverse_transform(preds_enc)

        # fill back
        df.loc[df['Brand'].isna(), 'Brand'] = preds

    return df

In [271]:
def impute_model(df, min_brand_count=20, includePrice=True):
    # Work on a copy so the original DataFrame is not modified
    df = df.copy()

    
    # --- Train a RandomForest classifier if there are still missing values ---
    # if still missing, train a classifier
    mode_lookup = (
    df.dropna(subset=['Brand', 'transmission', 'model'])
      .groupby(['Brand', 'transmission'])['model']
      .agg(lambda s: s.value_counts().idxmax())
    )
     
    for idx, row in df[df['model'].isna()].iterrows():
        brand = row['Brand']
        trans = row['transmission']
        key = (brand, trans)

        if key in mode_lookup.index:
            # häufigstes Modell einsetzen
            new_value = mode_lookup.loc[key]
            df.at[idx, 'model'] = new_value

    
    remaining = df['model'].isna().sum()
    
    if remaining > 0:
        # encode categorical predictors
        if includePrice:
            predictors = ['Brand', 'year', 'engineSize', 'mpg', 'tax', 'mileage', 'fuelType', 'transmission', 'price']
        else:
            predictors = ['Brand', 'year', 'engineSize', 'mpg', 'tax', 'mileage', 'fuelType', 'transmission']
        for col in predictors:
            if df[col].dtype == 'object':
                means = df.loc[df["model"].notna()].groupby(col)['price'].mean()
                df[col] = df[col].map(means) # for test data purposes, we only use infos from the traindata

        # split into known vs missing target
        fuelType_train = df[df['model'].notna()]
        test  = df[df['model'].isna()]

        X_train = fuelType_train[predictors]
        y_train = fuelType_train['model']
        X_test  = test[predictors]

        # encode target
        y_le = LabelEncoder()
        y_train_enc = y_le.fit_transform(y_train.astype(str))

        # train classifier
        clf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=77)
        clf.fit(X_train, y_train_enc)

        # predict and inverse-transform
        preds_enc = clf.predict(X_test)
        preds = y_le.inverse_transform(preds_enc)

        # fill back
        df.loc[df['model'].isna(), 'model'] = preds

    return df

In [272]:
%%time
X_train['price'] = y_train
X_train["transmission"] = impute_transmission(X_train)["transmission"]
X_train["fuelType"] = impute_fuelType(X_train)["fuelType"]
X_train["engineSize"] = impute_numeric_rf(X_train, "engineSize", ["fuelType","Brand", "transmission", "model"])["engineSize"]
X_train["mpg"] = impute_numeric_rf(X_train, "mpg", ['engineSize', 'fuelType', 'Brand', 'year', "tax", "model"])["mpg"]
X_train["tax"] = impute_numeric_rf(X_train, "tax", ['mpg', 'engineSize', 'year'])["tax"]
X_train["year"] = impute_numeric_rf(X_train, "year", ['Brand', 'engineSize', 'fuelType', 'mileage'])["year"].round().astype(int)
X_train["mileage"] = impute_numeric_rf(X_train, "mileage", ['year', 'engineSize', 'Brand', 'fuelType'])["mileage"]
X_train["paintQuality%"] = impute_numeric_rf(X_train, "paintQuality%", ["price", "year", "mileage","engineSize", "mpg", "Brand", "fuelType", "transmission"])["paintQuality%"]
X_train["previousOwners"] = impute_numeric_rf(X_train, "previousOwners", ["year", "price", "mileage", "engineSize", "Brand", "fuelType", "transmission"])["previousOwners"]
X_train["Brand"] = impute_brand(X_train)["Brand"]
X_train["hasDamage"] = X_train["hasDamage"].fillna(1)
X_train = X_train.drop(columns=["price"])

CPU times: total: 2min 34s
Wall time: 2min 35s


In [273]:
%%time
X_val['price'] = y_val
X_val["transmission"] = impute_transmission(X_val)["transmission"]
X_val["fuelType"] = impute_fuelType(X_val)["fuelType"]
X_val["engineSize"] = impute_numeric_rf(X_val, "engineSize", ["fuelType","Brand", "price", "transmission", "model"])["engineSize"]
X_val["mpg"] = impute_numeric_rf(X_val, "mpg", ['engineSize', 'fuelType', 'Brand', 'year', "tax", "model"])["mpg"]
X_val["tax"] = impute_numeric_rf(X_val, "tax", ['mpg', 'engineSize', 'year'])["tax"]
X_val["year"] = impute_numeric_rf(X_val, "year", ['price', 'Brand', 'engineSize', 'fuelType', 'mileage'])["year"].round().astype(int)
X_val["mileage"] = impute_numeric_rf(X_val, "mileage", ['year', 'price', 'engineSize', 'Brand', 'fuelType'])["mileage"]
X_val["paintQuality%"] = impute_numeric_rf(X_val, "paintQuality%", ["price", "year", "mileage","engineSize", "mpg", "Brand", "fuelType", "transmission"])["paintQuality%"]
X_val["previousOwners"] = impute_numeric_rf(X_val, "previousOwners", ["year", "price", "mileage", "engineSize", "Brand", "fuelType", "transmission"])["previousOwners"]
X_val["Brand"] = impute_brand(X_val)["Brand"]
X_val["hasDamage"] = X_val["hasDamage"].fillna(1)
X_val = X_val.drop(columns=["price"])

CPU times: total: 1min 6s
Wall time: 1min 6s


In [274]:
X_train["model"] = impute_model(X_train)["model"]
X_val["model"] = impute_model(X_val)["model"]

In [275]:
# quick check
# Look at the Missing values
missing_values = X_train.isnull().sum()

print("Missing values in train:")
print(missing_values[missing_values > 0]) 
print("\n")
missing_values = X_val.isnull().sum()

print("Missing values in validation:")
print(missing_values[missing_values > 0]) 
X_val.head()

Missing values in train:
Series([], dtype: int64)


Missing values in validation:
Series([], dtype: int64)


,Brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize,paintQuality%,previousOwners,hasDamage
69512,Mercedes-Benz,GLC,2018,Automatic,9500.0,Diesel,150.000000,56.500000,2.1,77.0,0.0,1.0
53000,Mercedes-Benz,C-Class,2017,Automatic,50369.0,Diesel,20.000000,65.700000,2.1,80.0,4.0,0.0
6366,Škoda,Citigo,2017,Manual,7489.0,Petrol,150.000000,64.200000,1.0,84.0,4.0,0.0
29021,Volkswagen,Touareg,2019,Automatic,7000.0,Diesel,145.000000,34.000000,3.0,62.0,0.0,0.0
10062,Ford,Focus,2019,Manual,10812.0,Diesel,145.680071,57.139354,1.5,99.0,2.0,0.0


In [276]:
train_info = X_train.copy()
val_info = X_val.copy()
train_info["price"] = y_train
val_info["price"] = y_val
train_val_infos = pd.concat([train_info, val_info])
train_val_infos["is_train"] = True
train_val_infos = train_val_infos.dropna()
train_val_infos["row_id"] = np.nan
X_test["is_train"]=False
X_test["price"]=np.nan
X_test["row_id"]=X_test.index
X_test.info()
train_val_infos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32567 entries, 0 to 32566
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   carID           32567 non-null  int64  
 1   Brand           31918 non-null  object 
 2   model           31917 non-null  object 
 3   year            31914 non-null  float64
 4   transmission    31944 non-null  object 
 5   mileage         31878 non-null  float64
 6   fuelType        31911 non-null  object 
 7   tax             29259 non-null  float64
 8   mpg             29279 non-null  float64
 9   engineSize      31939 non-null  float64
 10  paintQuality%   31942 non-null  float64
 11  previousOwners  31970 non-null  float64
 12  hasDamage       31970 non-null  float64
 13  is_train        32567 non-null  bool   
 14  price           0 non-null      float64
 15  row_id          32567 non-null  int64  
dtypes: bool(1), float64(9), int64(2), object(4)
memory usage: 3.8+ MB
<class 'pa

In [277]:
%%time
# compute transmission
compute_transmission = pd.concat([train_val_infos.copy(), X_test.loc[X_test["transmission"].isna()]])
transmission_values = impute_transmission(compute_transmission).loc[compute_transmission["is_train"]== False]
for idx, row in X_test[X_test['transmission'].isna()].iterrows():
    row_id = row['row_id']
    new_value = transmission_values.loc[transmission_values['row_id']==row_id, 'transmission'].iloc[0]
    X_test.at[idx, 'transmission'] = new_value

CPU times: total: 8.86 s
Wall time: 8.89 s


In [278]:
%%time
# compute fuelType
compute_fuelType = pd.concat([train_val_infos.copy(), X_test.loc[X_test["fuelType"].isna()]])
fuelType_values = impute_fuelType(compute_fuelType).loc[compute_fuelType["is_train"]== False]
for idx, row in X_test[X_test['fuelType'].isna()].iterrows():
    row_id = row['row_id']
    new_value = fuelType_values.loc[fuelType_values['row_id']==row_id, 'fuelType'].iloc[0]
    X_test.at[idx, 'fuelType'] = new_value


CPU times: total: 11.8 s
Wall time: 11.8 s


In [279]:
%%time
# compute engineSize
compute_engineSize = pd.concat([train_val_infos.copy(), X_test.loc[X_test["engineSize"].isna()]])
engineSize_values = impute_numeric_rf(compute_engineSize, "engineSize", ["fuelType","Brand", "transmission", "model"]).loc[compute_engineSize["is_train"]== False]
for idx, row in X_test[X_test['engineSize'].isna()].iterrows():
    row_id = row['row_id']
    new_value = engineSize_values.loc[engineSize_values['row_id']==row_id, 'engineSize'].iloc[0]
    X_test.at[idx, 'engineSize'] = new_value

CPU times: total: 8.5 s
Wall time: 8.5 s


In [280]:
# fill has damage
X_test["hasDamage"] = X_test["hasDamage"].fillna(1)

In [281]:
%%time
# compute mpg
compute_mpg = pd.concat([train_val_infos.copy(), X_test.loc[X_test["mpg"].isna()]])
mpg_values = impute_numeric_rf(compute_mpg, "mpg", ['engineSize', 'fuelType', 'Brand', 'year', "tax", "model"]).loc[compute_mpg["is_train"]== False]
for idx, row in X_test[X_test['mpg'].isna()].iterrows():
    row_id = row['row_id']
    new_value = mpg_values.loc[mpg_values['row_id']==row_id, 'mpg'].iloc[0]
    X_test.at[idx, 'mpg'] = new_value

CPU times: total: 18.5 s
Wall time: 18.8 s


In [282]:
%%time
# compute tax
compute_tax = pd.concat([train_val_infos.copy(), X_test.loc[X_test["tax"].isna()]])
tax_values = impute_numeric_rf(compute_tax, "tax", ['mpg', 'engineSize', 'year']).loc[compute_tax["is_train"]== False]
for idx, row in X_test[X_test['tax'].isna()].iterrows():
    row_id = row['row_id']
    new_value = tax_values.loc[tax_values['row_id']==row_id, 'tax'].iloc[0]
    X_test.at[idx, 'tax'] = new_value

CPU times: total: 12.9 s
Wall time: 13 s


In [283]:
%%time
# compute year
compute_year = pd.concat([train_val_infos.copy(), X_test.loc[X_test["year"].isna()]])
year_values = impute_numeric_rf(compute_year, "year", ['Brand', 'engineSize', 'fuelType', 'mileage']).loc[compute_year["is_train"]== False]
for idx, row in X_test[X_test['year'].isna()].iterrows():
    row_id = row['row_id']
    new_value = year_values.loc[year_values['row_id']==row_id, 'year'].iloc[0]
    X_test.at[idx, 'year'] = new_value

CPU times: total: 27.8 s
Wall time: 28 s


In [284]:
%%time
# compute mileage
compute_mileage = pd.concat([train_val_infos.copy(), X_test.loc[X_test["mileage"].isna()]])
mileage_values = impute_numeric_rf(compute_mileage, "mileage", ['year', 'engineSize', 'Brand', 'fuelType']).loc[compute_mileage["is_train"]== False]
for idx, row in X_test[X_test['mileage'].isna()].iterrows():
    row_id = row['row_id']
    new_value = mileage_values.loc[mileage_values['row_id']==row_id, 'mileage'].iloc[0]
    X_test.at[idx, 'mileage'] = new_value

CPU times: total: 9.55 s
Wall time: 9.55 s


In [285]:
%%time
# compute paintQuality%
compute_paintQuality = pd.concat([train_val_infos.copy(), X_test.loc[X_test["paintQuality%"].isna()]])
paintQuality_values = impute_numeric_rf(compute_paintQuality, "paintQuality%", ["price", "year", "mileage","engineSize", "mpg", "Brand", "fuelType", "transmission"]).loc[compute_paintQuality["is_train"]== False]
for idx, row in X_test[X_test['paintQuality%'].isna()].iterrows():
    row_id = row['row_id']
    new_value = paintQuality_values.loc[paintQuality_values['row_id']==row_id, 'paintQuality%'].iloc[0]
    X_test.at[idx, 'paintQuality%'] = new_value

CPU times: total: 1min 1s
Wall time: 1min 1s


In [286]:
%%time
# compute previousOwners
compute_previousOwners = pd.concat([train_val_infos.copy(), X_test.loc[X_test["previousOwners"].isna()]])
previousOwners_values = impute_numeric_rf(compute_previousOwners, "previousOwners", ["year", "price", "mileage", "engineSize", "Brand", "fuelType", "transmission"]).loc[compute_previousOwners["is_train"]== False]
for idx, row in X_test[X_test['previousOwners'].isna()].iterrows():
    row_id = row['row_id']
    new_value = previousOwners_values.loc[previousOwners_values['row_id']==row_id, 'previousOwners'].iloc[0]
    X_test.at[idx, 'previousOwners'] = new_value

CPU times: total: 50.4 s
Wall time: 50.8 s


In [287]:
%%time
# compute Brand
compute_brand = pd.concat([train_val_infos.copy(), X_test.loc[X_test["Brand"].isna()]])
brand_values = impute_brand(compute_brand).loc[compute_brand["is_train"]== False]
for idx, row in X_test[X_test['Brand'].isna()].iterrows():
    row_id = row['row_id']
    new_value = brand_values.loc[brand_values['row_id']==row_id, 'Brand'].iloc[0]
    X_test.at[idx, 'Brand'] = new_value

CPU times: total: 10.3 s
Wall time: 10.4 s


In [288]:
%%time
# compute model
compute_model = pd.concat([train_val_infos.copy(), X_test.loc[X_test["model"].isna()]])
model_values = impute_model(compute_model).loc[compute_model["is_train"]== False]
for idx, row in X_test[X_test['model'].isna()].iterrows():
    row_id = row['row_id']
    new_value = model_values.loc[model_values['row_id']==row_id, 'model'].iloc[0]
    X_test.at[idx, 'model'] = new_value

CPU times: total: 46.6 s
Wall time: 46.9 s


In [289]:
X_test = X_test.drop(columns=["price", "row_id", "is_train"])

In [290]:
# quick check
# Look at the Missing values
missing_values = X_test.isnull().sum()

print("Missing values in test:")
print(missing_values[missing_values > 0]) 

Missing values in test:
Series([], dtype: int64)


# Feature Engineering

In [291]:
# age affects cars much more than the year: car_age in years
X_train['car_age'] = 2020 - X_train['year']
# it is important how efficent the motor is, 
X_train['efficiency_ratio'] = X_train['mpg'] / X_train['engineSize']

X_train['mileage_per_year'] = X_train['mileage'] / (X_train['car_age'] + 1)

X_train['diesel_efficiency'] = (X_train['fuelType'] == 'Diesel') * X_train['efficiency_ratio']

X_train['petrol_efficiency'] = (X_train['fuelType'] == 'Petrol') * X_train['efficiency_ratio']

brand_popularity = X_train['Brand'].value_counts(normalize=True)
X_train['brand_popularity'] = X_train['Brand'].map(brand_popularity)

X_train['owners_flag'] = (X_train['previousOwners'] > 2).astype(int)

X_train['previousOwners_sq'] = X_train['previousOwners'] ** 2

X_train['engine_tax_ratio'] = X_train['engineSize'] / (X_train['tax'] + 1)

In [292]:
# age affects cars much more than the year: car_age in years
X_val['car_age'] = 2020 - X_val['year']
# it is important how efficent the motor is, 
X_val['efficiency_ratio'] = X_val['mpg'] / X_val['engineSize']

X_val['mileage_per_year'] = X_val['mileage'] / (X_val['car_age'] + 1)

X_val['diesel_efficiency'] = (X_val['fuelType'] == 'Diesel') * X_val['efficiency_ratio']

X_val['petrol_efficiency'] = (X_val['fuelType'] == 'Petrol') * X_val['efficiency_ratio']

brand_popularity = X_val['Brand'].value_counts(normalize=True)
X_val['brand_popularity'] = X_val['Brand'].map(brand_popularity)

X_val['owners_flag'] = (X_val['previousOwners'] > 2).astype(int)

X_val['previousOwners_sq'] = X_val['previousOwners'] ** 2

X_val['engine_tax_ratio'] = X_val['engineSize'] / (X_val['tax'] + 1)

In [293]:
# age affects cars much more than the year: car_age in years
X_test['car_age'] = 2020 - X_test['year']
# it is important how efficent the motor is, 
X_test['efficiency_ratio'] = X_test['mpg'] / X_test['engineSize']

X_test['mileage_per_year'] = X_test['mileage'] / (X_test['car_age'] + 1)

X_test['diesel_efficiency'] = (X_test['fuelType'] == 'Diesel') * X_test['efficiency_ratio']

X_test['petrol_efficiency'] = (X_test['fuelType'] == 'Petrol') * X_test['efficiency_ratio']

brand_popularity = X_test['Brand'].value_counts(normalize=True)
X_test['brand_popularity'] = X_test['Brand'].map(brand_popularity)

X_test['owners_flag'] = (X_test['previousOwners'] > 2).astype(int)

X_test['previousOwners_sq'] = X_test['previousOwners'] ** 2

X_test['engine_tax_ratio'] = X_test['engineSize'] / (X_test['tax'] + 1)

# Encoding categorical variables

- We have to find out which encoding method is the best for our categorical features

In [294]:
X_train['price'] = y_train
X_train['brand_encoded'] = X_train.groupby('Brand')['price'].transform('mean')
X_train['model_delta_encoded'] = X_train.groupby('model')['price'].transform('mean') - X_train['brand_encoded']
X_train['transmission_encoded'] =  X_train.groupby('transmission')['price'].transform('mean')
X_train['fuelType_encoded'] =  X_train.groupby('fuelType')['price'].transform('mean')
X_train = X_train.drop(columns=["price"])

In [295]:
X_val['price'] = y_val
X_val['brand_encoded'] = X_val.groupby('Brand')['price'].transform('mean')
X_val['model_delta_encoded'] = X_val.groupby('model')['price'].transform('mean') - X_val['brand_encoded']
X_val['transmission_encoded'] =  X_val.groupby('transmission')['price'].transform('mean')
X_val['fuelType_encoded'] =  X_val.groupby('fuelType')['price'].transform('mean')
X_val = X_val.drop(columns=["price"])

In [297]:
X_train['price'] = y_train
brand_mean_price = X_train.groupby('Brand')['price'].mean()
X_test['brand_encoded'] = X_test['Brand'].map(brand_mean_price)
global_mean = X_train['price'].mean()
X_test['brand_encoded'].fillna(global_mean, inplace=True)

model_delta_mean = X_train.groupby('model')['price'].transform('mean') - X_train['brand_encoded']
X_test['model_delta_encoded'] = X_test['model'].map(model_delta_mean)
X_test['model_delta_encoded'].fillna(0, inplace=True)

transmission_mean_price = X_train.groupby('transmission')['price'].mean()
X_test['transmission_encoded'] = X_test['transmission'].map(transmission_mean_price)
global_mean = X_train['price'].mean()
X_test['transmission_encoded'].fillna(global_mean, inplace=True)

fuelType_mean_price = X_train.groupby('fuelType')['price'].mean()
X_test['fuelType_encoded'] = X_test['fuelType'].map(fuelType_mean_price)
global_mean = X_train['price'].mean()
X_test['fuelType_encoded'].fillna(global_mean, inplace=True)
X_train = X_train.drop(columns=["price"])

# Not yet: Scaling

After encoding, all numerical features were scaled using RobustScaler.
This approach reduces the influence of outliers by centering features around the median and scaling them by the interquartile range (IQR).
The scaler was fitted only on the training set to prevent data leakage.
The column carID was dropped since it serves as an identifier and carries no predictive information.


- In the future we can consider scaling the target variable price y

In [298]:
# ======================================================
# 🔢 Feature Scaling (RobustScaler, no data leakage)
# ======================================================
"""
Reasoning:
- Numerical features (e.g., mileage, tax, engineSize) vary widely in scale.
- Since the dataset includes outliers (e.g., price and mileage extremes),
  we use RobustScaler, which scales using the median and IQR instead of mean/std.
- The scaling is fit only on X_train to avoid data leakage.
- carID is dropped because it's only an identifier and not informative.
"""

#from sklearn.preprocessing import RobustScaler
#import pandas as pd

# 1️⃣ Remove non-informative ID column
#drop_cols = ["carID"]
#for df_name, df in [("X_train_enc", X_train_enc), ("X_val_enc", X_val_enc), ("x_test_enc", x_test_enc)]:
#    if "carID" in df.columns:
#        df.drop(columns=drop_cols, inplace=True)
#        print(f"Removed {drop_cols} from {df_name}")

# 2️⃣ Update list of numeric columns (exclude ID)
#numeric_cols = [c for c in X_train_enc.select_dtypes(include=["number"]).columns if c not in drop_cols]
#print("Numeric columns to be scaled:", numeric_cols)

# 3️⃣ Initialize and fit scaler only on training set
#scaler = RobustScaler()
#scaler.fit(X_train_enc[numeric_cols])

# 4️⃣ Apply scaling to all splits (leakage-free)
#X_train_scaled = X_train_enc.copy()
#X_val_scaled   = X_val_enc.copy()
#x_test_scaled  = x_test_enc.copy()

#X_train_scaled[numeric_cols] = scaler.transform(X_train_enc[numeric_cols])
#X_val_scaled[numeric_cols]   = scaler.transform(X_val_enc[numeric_cols])
#x_test_scaled[numeric_cols]  = scaler.transform(x_test_enc[numeric_cols])

# 5️⃣ Sanity check
#print("Scaling complete ✅")
#print("Example preview (scaled numeric columns):")
#display(X_train_scaled[numeric_cols].head())


"\nReasoning:\n- Numerical features (e.g., mileage, tax, engineSize) vary widely in scale.\n- Since the dataset includes outliers (e.g., price and mileage extremes),\n  we use RobustScaler, which scales using the median and IQR instead of mean/std.\n- The scaling is fit only on X_train to avoid data leakage.\n- carID is dropped because it's only an identifier and not informative.\n"

# Output Save

In [299]:
# ======================================================
# 💾 Save Processed Datasets
# ======================================================
"""
Save X_train, y_train, X_val, y_val, and X_test separately.
This structure is cleaner for later model loading and avoids re-splitting.
"""

import os

output_dir = os.path.join(data_dir, "encoded_data")
os.makedirs(output_dir, exist_ok=True)

# Save feature and target sets separately
X_train.to_csv(os.path.join(output_dir, "12_X_train.csv"), index=False)
y_train.to_csv(os.path.join(output_dir, "12_y_train.csv"), index=False)

X_val.to_csv(os.path.join(output_dir, "12_X_val.csv"), index=False)
y_val.to_csv(os.path.join(output_dir, "12_y_val.csv"), index=False)

X_test.to_csv(os.path.join(output_dir, "12_X_test.csv"), index=False)

print("✅ Processed data saved successfully (X/y separated):")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test:  {X_test.shape}")


✅ Processed data saved successfully (X/y separated):
X_train: (56979, 25), y_train: (56979,)
X_val:   (18994, 25), y_val: (18994,)
X_test:  (32567, 26)
